# 🎬 CineMatch – Entrenamiento del Modelo Colaborativo (SVD)

## 🚀 Overview

Este notebook implementa el proceso de **entrenamiento del modelo de filtrado colaborativo basado en SVD (Singular Value Decomposition)**, que constituye el núcleo del sistema de recomendación híbrido.

El objetivo es aprender **patrones latentes entre usuarios y películas**, a partir de sus interacciones (ratings), permitiendo generar recomendaciones altamente personalizadas.

El modelo entrenado se exporta como:

```text
svd_collaborative_model_v1.pth
```

y será utilizado posteriormente en el sistema híbrido.

---

## 🧠 Problema a resolver

Los métodos tradicionales basados en contenido (géneros, popularidad) tienen limitaciones:

* No capturan relaciones complejas entre usuarios
* No aprenden patrones implícitos
* No personalizan suficientemente

👉 El filtrado colaborativo soluciona esto aprendiendo de:

> “Usuarios con comportamientos similares tienden a consumir contenido similar”

---

## ⚙️ Enfoque del modelo

Se utiliza un modelo tipo **Matrix Factorization (SVD)** implementado con PyTorch:

* Cada usuario → vector embedding
* Cada película → vector embedding
* La predicción se basa en la interacción entre ambos

---

## 🧩 Arquitectura del modelo

El modelo aprende:

* 🎯 `user_embedding` → representación latente del usuario
* 🎬 `movie_embedding` → representación latente de la película
* ⚖️ `user_bias` → sesgo del usuario
* ⚖️ `movie_bias` → sesgo de la película

La predicción se calcula como:

```text
score = dot(user_vector, movie_vector) + user_bias + movie_bias
```

---

## 📊 Datos de entrada

Se utiliza el dataset de ratings:

### 📁 `ratings_df`

Contiene:

* `userId`
* `movieId`
* `rating`

---

## 🔄 Flujo completo del entrenamiento

El pipeline sigue los siguientes pasos:

---

### 1️⃣ Preparación de datos

* Conversión de IDs a índices numéricos
* Creación de diccionarios:

  * `user_to_index`
  * `movie_to_index`

---

### 2️⃣ Creación del modelo

* Inicialización de embeddings
* Definición de arquitectura en PyTorch

---

### 3️⃣ Entrenamiento

* Forward pass → predicción de ratings
* Cálculo de pérdida (MSE)
* Backpropagation
* Optimización de parámetros

---

### 4️⃣ Evaluación (opcional)

* Métricas como RMSE
* Validación del modelo

---

### 5️⃣ Exportación del modelo

Se guarda un checkpoint con:

* Pesos del modelo
* Diccionarios de mapeo
* Media global de ratings
* Dimensión de embeddings

---

## 📦 Output del notebook

El resultado final es el fichero:

```text
svd_collaborative_model_v1.pth
```

Este contiene todo lo necesario para inferencia:

* Modelo entrenado
* Estructura de embeddings
* Mapeos usuario/película

---

## 🔗 Integración con el sistema híbrido

Este modelo será utilizado por:

```text
hybrid_recommender.py
```

y se activará cuando:

* El usuario tenga suficiente historial
* El usuario esté presente en el modelo

---

## 🎯 Objetivo del notebook

* Entrenar el modelo colaborativo
* Generar el fichero `.pth`
* Validar que el modelo aprende correctamente
* Preparar el sistema para inferencia en producción

---

## 💡 Conclusión

Este modelo permite capturar relaciones complejas entre usuarios y contenido:

> No se basa en lo que las películas son, sino en cómo las personas interactúan con ellas.

Es la base de las recomendaciones personalizadas en el sistema híbrido.

---


## Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

## Carga de datos

In [2]:
def load_ratings(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df[["userId", "movieId", "rating"]].copy()

    df["userId"] = df["userId"].astype(np.int64)
    df["movieId"] = df["movieId"].astype(np.int64)
    df["rating"] = df["rating"].astype(np.float32)

    return df


def split_data(ratings_df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    train_df, test_df = train_test_split(
        ratings_df,
        test_size=test_size,
        random_state=random_state
    )
    return train_df, test_df

### Mapeos

In [3]:
def create_mappings(ratings_df: pd.DataFrame):
    user_ids = ratings_df["userId"].unique()
    movie_ids = ratings_df["movieId"].unique()

    user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
    movie_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

    return user_to_index, movie_to_index

### Dataset Optimizado

In [4]:
class RatingsDataset(Dataset):
    def __init__(self, df, user_map, movie_map):
        self.users = df["userId"].map(user_map).to_numpy(np.int64)
        self.movies = df["movieId"].map(movie_map).to_numpy(np.int64)
        self.ratings = df["rating"].to_numpy(np.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

### Modelo

In [5]:
class MovieRecommender(nn.Module):
    def __init__(self, num_users: int, num_movies: int, embedding_dim: int = 128):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)

        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.movie_embedding.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, users, movies):
        user_vec = self.user_embedding(users)
        movie_vec = self.movie_embedding(movies)

        user_b = self.user_bias(users).squeeze(1)
        movie_b = self.movie_bias(movies).squeeze(1)

        interaction = (user_vec * movie_vec).sum(dim=1)

        x = interaction + user_b + movie_b
        x = torch.sigmoid(x)

        return x * 4.5 + 0.5

### Entrenamiento optimizado

In [6]:
def train_model(
    model,
    dataloader,
    epochs: int = 8,
    lr: float = 0.002,
    weight_decay: float = 1e-5,
    device: str = "cpu"
):
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for users, movies, ratings in dataloader:
            users = users.to(device, non_blocking=True)
            movies = movies.to(device, non_blocking=True)
            ratings = ratings.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast("cuda"):
                    predictions = model(users, movies)
                    loss = criterion(predictions, ratings)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                predictions = model(users, movies)
                loss = criterion(predictions, ratings)

                loss.backward()
                optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")

### Evualuacion optimizada por batches

In [7]:
def evaluate_model_fast(
    model,
    test_df: pd.DataFrame,
    user_to_index: dict,
    movie_to_index: dict,
    device: str = "cpu",
    batch_size: int = 65536
):
    model.eval()
    model.to(device)

    valid_df = test_df[
        test_df["userId"].isin(user_to_index) &
        test_df["movieId"].isin(movie_to_index)
    ].copy()

    user_indices = valid_df["userId"].map(user_to_index).to_numpy(dtype=np.int64)
    movie_indices = valid_df["movieId"].map(movie_to_index).to_numpy(dtype=np.int64)
    targets = valid_df["rating"].to_numpy(dtype=np.float32)

    users_tensor = torch.tensor(user_indices, dtype=torch.long)
    movies_tensor = torch.tensor(movie_indices, dtype=torch.long)
    targets_tensor = torch.tensor(targets, dtype=torch.float32)

    preds = []

    with torch.no_grad():
        for start in range(0, len(users_tensor), batch_size):
            end = start + batch_size

            u_batch = users_tensor[start:end].to(device, non_blocking=True)
            m_batch = movies_tensor[start:end].to(device, non_blocking=True)

            batch_preds = model(u_batch, m_batch)
            preds.append(batch_preds.cpu())

    preds = torch.cat(preds).numpy()
    targets = targets_tensor.numpy()

    rmse = np.sqrt(np.mean((preds - targets) ** 2))
    mae = np.mean(np.abs(preds - targets))

    return rmse, mae

### Main para ratings_clean

In [8]:
ratings_df = load_ratings("../../../data/processed/ratings_clean.csv")

train_df, test_df = split_data(ratings_df, test_size=0.2, random_state=42)

user_to_index, movie_to_index = create_mappings(train_df)

train_dataset = RatingsDataset(
    train_df,
    user_to_index,
    movie_to_index
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=131072,
    shuffle=True,
    num_workers=8,      # en notebook mejor 0
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

model = MovieRecommender(
    num_users=len(user_to_index),
    num_movies=len(movie_to_index),
    embedding_dim=192
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando device:", device)

train_model(
    model=model,
    dataloader=train_dataloader,
    epochs=8,
    lr=0.002,
    weight_decay=1e-5,
    device=device
)

rmse, mae = evaluate_model_fast(
    model=model,
    test_df=test_df,
    user_to_index=user_to_index,
    movie_to_index=movie_to_index,
    device=device,
    batch_size=131072
)

print(f"\nRMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

global_mean = ratings_df["rating"].mean()

torch.save({
    "model_state_dict": model.state_dict(),
    "user_to_index": user_to_index,
    "movie_to_index": movie_to_index,
    "embedding_dim": 192,
    "global_mean": global_mean
}, "recommender_full.pth")

Usando device: cuda
Epoch 1/8 - Loss: 1.0264
Epoch 2/8 - Loss: 0.7648
Epoch 3/8 - Loss: 0.7341
Epoch 4/8 - Loss: 0.7130
Epoch 5/8 - Loss: 0.6994
Epoch 6/8 - Loss: 0.6904
Epoch 7/8 - Loss: 0.6842
Epoch 8/8 - Loss: 0.6798

RMSE: 0.8306
MAE: 0.6416
